# KrishiDisha teacher distillation on a Kaggle CPU kernel

Runs the teacher pool (Groq, Gemini, OpenRouter, optional Ollama cloud) for the trajectory and Hindi-rewrite stages, resuming from the private cache dataset and re-uploading the cache after every pass, for up to `MAX_HOURS`. Secrets needed (Add-ons > Secrets): `GEMINI_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (to version the cache dataset), optional `OLLAMA_API_KEY`.


In [ ]:
# Source comes from the private Kaggle dataset (git archive of the lab repo), not from GitHub.
# Kaggle extracts uploaded zips, so the tree is at .../krishidisha-lab-src/src/krishidisha; fall back to src.zip.
import glob, os, shutil, zipfile
shutil.rmtree("/tmp/krishidisha", ignore_errors=True)
_dirs = glob.glob("/kaggle/input/**/krishidisha-lab-src/src/krishidisha", recursive=True)
if _dirs:
    shutil.copytree(_dirs[0], "/tmp/krishidisha")
    _ver = os.path.join(os.path.dirname(os.path.dirname(_dirs[0])), "VERSION.txt")
else:
    _zip = glob.glob("/kaggle/input/**/krishidisha-lab-src/**/src.zip", recursive=True)[0]
    zipfile.ZipFile(_zip).extractall("/tmp")
    _ver = os.path.join(os.path.dirname(_zip), "VERSION.txt")
print("source version:", open(_ver).read().strip() if os.path.exists(_ver) else "unknown", "->", os.listdir("/tmp/krishidisha")[:8])

import os
W = "/kaggle/working"
os.chdir("/tmp/krishidisha")
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
_names = ["GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY", "OLLAMA_API_KEY"]
_names += [f"{b}_{i}" for b in ("GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY") for i in range(2, 6)]
for k in _names:
    try:
        os.environ[k] = _s.get_secret(k)
    except Exception:
        pass
print("secrets attached:", [k for k in _names if os.environ.get(k)])


In [ ]:
# All logic lives in the lab source so updates never need an edit here
exec(open("scripts/kaggle_distill_driver.py", encoding="utf-8").read())
